#import payspark

In [0]:
from pyspark.sql.functions import *

# create a widgit for incremental run

In [0]:
dbutils.widgets.text("incremental flag","0")

# check incremental run

In [0]:
incremental_flag = dbutils.widgets.get("incremental flag")
incremental_flag

# Establish connection with ADLS

In [0]:
# Azure Storage authentication
# Configure Azure Storage credentials securely before execution.
# Do not store access keys or secrets in this notebook.

# check connection
- abfss - azure blob file system storage

In [0]:
%sql
select * from parquet.`abfss://silver@cfstorageaccountrkd.dfs.core.windows.net/silverdata`


# Working on Region Dimension

In [0]:
%sql
select distinct(SalesRegion) as SalesRegion from parquet.`abfss://silver@cfstorageaccountrkd.dfs.core.windows.net/silverdata`


# Create a DataFrame - df_src
- src = source 
- df = dataframe

In [0]:
df_src = spark.sql(
    """
    select distinct(SalesRegion) as SalesRegion 
    from parquet.`abfss://silver@cfstorageaccountrkd.dfs.core.windows.net/silverdata`
    """
)
df_src.display()

In [0]:
# To check Schema # 

df_src.printSchema()

# Getting Gold Layer Data - df_sink

In [0]:
from delta.tables import DeltaTable

path = 'abfss://gold@cfstorageaccountrkd.dfs.core.windows.net/dim_region'

if DeltaTable.isDeltaTable(spark, path):

#if spark.catalog.tableExists("gold.dim_region"):
    df_sink = spark.sql(
    """
    select 
        Dim_Region_Key, 
        SalesRegion 
    from delta.`abfss://gold@cfstorageaccountrkd.dfs.core.windows.net/dim_region` 
    """
    )
    print('Good!! Table Exist')
else:
    df_sink = spark.createDataFrame([],schema="Dim_Region_Key int, SalesRegion string")

    df_sink.display()


# Old and New Records - Left Join

In [0]:
df = df_src.join(
                 df_sink, df_src['SalesRegion'] == df_sink['SalesRegion'],'left').select(df_src['SalesRegion'],   
                 df_sink['Dim_Region_Key']
                 )

df.display()

# Old Records

In [0]:
df_old = df.filter(df['Dim_Region_Key'].isNotNull())
df_old.display()


# New Record

In [0]:
df_new = df.filter(df['Dim_Region_Key'].isNull())
df_new.display()

# Get Maximum Dim_Region_Key

In [0]:
if incremental_flag == '0':
    max_value = 1
else:
    max_value = df_old.agg(max(df_old['Dim_Region_Key'])).collect()[0][0] + 1

max_value 



# Creating Surrogate Key

In [0]:
df = df_new.withColumn('Dim_Region_Key',max_value + monotonically_increasing_id())
df.display()

# Union Old and New

In [0]:
df = df_old.union(df)
df.display()

#SCD Type 1 - Upsert
- SCD (Slowly Changing Dimenstion)
- ACID Transaction - Atomicity Consistency Isolation Durability

In [0]:

from delta.tables import DeltaTable

table_name = "gold.dim_region"

path = "abfss://gold@cfstorageaccountrkd.dfs.core.windows.net/dim_region"

if DeltaTable.isDeltaTable(spark, path):

    deltaTable = DeltaTable.forPath(spark, path)

    deltaTable.alias("target").merge(
        df.alias("source"),
        "target.Dim_Region_Key = source.Dim_Region_Key"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

    print("Good! Data merged successfully")

else:

    df.write.format("delta") \
        .mode("overwrite") \
        .option("mergeSchema", "true") \
        .save(path)

    print("The table is created")


In [0]:
# from delta.tables import DeltaTable 

# table_name = 'gold.dim_region'
# path = 'abfss://gold@cfstorageaccountrkd.dfs.core.windows.net/dim_region'

# if DeltaTable.isDeltaTable(spark,path):
# # if spark.catalog.tableExists(table_name):
#     deltaTable = DeltaTable.forPath(spark,path)
#     deltaTable.alias('target').merge(df.alias('source'),
#     'target.Dim_Region_Key = source_Dim_Region_Key').whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
#     print('good')
# else:
#     df.write.format('delta').mode('overwrite').option('meargeSchema','true').save(path)
# print('The table is created')

In [0]:
%sql
select * from delta.`abfss://gold@cfstorageaccountrkd.dfs.core.windows.net/dim_region`